# Trabalho Prático 01 — resolução de referência

**Documento do professor.** Este notebook resolve, tarefa por tarefa, o enunciado do `Trabalho pratico 01.pdf`, com as sementes que ele fixa. É daqui que saem os números do roteiro de correção. Ele não é o único caminho certo: onde o enunciado deixa uma escolha ao aluno, a escolha feita aqui é uma entre várias razoáveis, e o texto diz qual.

A numeração das seções é a das tarefas: R1 a R8 para a regressão, C1 a C7 para a classificação.

## Pacotes

Os mesmos das aulas práticas. `warnings` fica silenciado pelo motivo de sempre (avisos de convergência e de versão poluem a saída); onde um aviso importa, ele é tratado explicitamente.

In [ ]:
import os
import time
import warnings

import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots
from scipy import stats

import sklearn.linear_model as skl
import sklearn.model_selection as skm
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (StandardScaler, OneHotEncoder, PolynomialFeatures,
                                   SplineTransformer, FunctionTransformer)
from sklearn.dummy import DummyRegressor
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (mean_squared_error, confusion_matrix, roc_curve, roc_auc_score,
                             precision_recall_curve, average_precision_score,
                             brier_score_loss, log_loss)

warnings.filterwarnings("ignore")
inicio = time.time()

# Parte I — Regressão: a resistência do concreto

## R1. Conhecendo os dados

O arquivo é procurado primeiro na pasta deste notebook e depois em `../recursos/dados/`, sem baixar nada da rede.

In [ ]:
_nome = "concreto.csv"

# procura em dois lugares, sem baixar nada da internet: a pasta deste
# notebook primeiro ou então ../recursos/dados/
_lugares = [_nome, os.path.join("..", "recursos", "dados", _nome)]
_caminho = next((c for c in _lugares if os.path.exists(c)), None)

if _caminho is None:
    raise FileNotFoundError(
        f"Não encontrei {_nome}. Procurei em: {', '.join(_lugares)}. "
        "Coloque o arquivo na mesma pasta deste notebook.")

concreto = pd.read_csv(_caminho)
print(concreto.shape)
concreto.describe().T.round(2)

**(a)** São $n=1030$ linhas e $p=8$ covariáveis, todas quantitativas, sem faltantes. A resposta, resistência à compressão em MPa, é quantitativa: faz sentido falar em "resistência média", e o problema é de **regressão**.

In [ ]:
print("faltantes:", concreto.isna().sum().sum())

**(b)** A resistência contra a idade e contra o cimento. A idade é a relação mais visivelmente não linear: a resistência sobe depressa nas primeiras semanas e quase estabiliza depois de 90 dias — e os ensaios se concentram em poucas idades (3, 7, 14, 28, 56, 90...).

In [ ]:
fig, ax = subplots(1, 3, figsize=(15, 4))
ax[0].hist(concreto["resistencia"], bins=30)
ax[0].set_xlabel("resistência (MPa)"); ax[0].set_ylabel("frequência")
ax[1].scatter(concreto["idade"], concreto["resistencia"], s=8, alpha=0.5)
ax[1].set_xscale("log"); ax[1].set_xlabel("idade (dias, escala log)"); ax[1].set_ylabel("resistência (MPa)")
ax[2].scatter(concreto["agua"] / concreto["cimento"], concreto["resistencia"], s=8, alpha=0.5)
ax[2].set_xlabel("razão água/cimento"); ax[2].set_ylabel("resistência (MPa)")
fig.tight_layout()
print(concreto["idade"].value_counts().sort_index().to_dict())

**(c)** Linhas duplicadas. Uma medida contínua repetida até a oitava casa decimal não é um segundo ensaio: é o mesmo registro digitado duas vezes.

In [ ]:
dup = concreto.duplicated()
print("linhas idênticas a uma anterior:", dup.sum())
covs = concreto.columns[:8].tolist()
print("linhas com covariáveis repetidas:", concreto.duplicated(subset=covs).sum())

**(d)** As réplicas verdadeiras — mesmas oito covariáveis, resistências diferentes — permitem estimar o erro irredutível sem modelo nenhum, pela variância combinada dentro dos grupos. Com as duplicatas, os zeros artificiais puxam a estimativa para baixo; sem elas, sobram poucos graus de liberdade, e o intervalo (qui-quadrado, sob normalidade e homocedasticidade) mostra quão pouco elas dizem.

In [ ]:
def variancia_pura(d):
    r = d.groupby(covs)["resistencia"].agg(["count", "var"])
    r = r[r["count"] > 1]
    gl = (r["count"] - 1).sum()
    s2 = ((r["count"] - 1) * r["var"]).sum() / gl
    return len(r), r["count"].sum(), gl, s2

for rotulo, d in [("com duplicatas", concreto), ("sem duplicatas", concreto.drop_duplicates())]:
    grupos, obs, gl, s2 = variancia_pura(d)
    ic = (gl * s2 / stats.chi2.ppf(0.975, gl), gl * s2 / stats.chi2.ppf(0.025, gl))
    print(f"{rotulo:15s} grupos {grupos:2d}  obs {obs:2d}  gl {gl:2d}  s2 = {s2:6.2f}  IC95% = ({ic[0]:.1f}; {ic[1]:.1f})")

**(e)** O traço. O mesmo traço (as sete quantidades de ingredientes) foi ensaiado em várias idades: 1030 linhas vêm de 428 traços.

In [ ]:
concreto = concreto.drop_duplicates().reset_index(drop=True)
componentes = concreto.columns[:7].tolist()
concreto["traco"] = concreto.groupby(componentes).ngroup()
print("linhas:", len(concreto), " traços:", concreto["traco"].nunique())
print(concreto["traco"].value_counts().value_counts().sort_index().to_dict())

## R2. Protocolo de avaliação

**(a)** O teste é separado **por traço**, com a semente do enunciado. Nenhum traço aparece dos dois lados.

In [ ]:
X = concreto[covs]
y = concreto["resistencia"].to_numpy()
grupos = concreto["traco"].to_numpy()

separa = skm.GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=2026)
i_tr, i_te = next(separa.split(X, y, grupos))
X_tr, X_te, y_tr, y_te = X.iloc[i_tr], X.iloc[i_te], y[i_tr], y[i_te]
g_tr = grupos[i_tr]
print("treino:", len(i_tr), "teste:", len(i_te))
print("traços em comum:", len(set(grupos[i_tr]) & set(grupos[i_te])))

**(c)** Toda escolha de hiperparâmetro usa estas dobras, sempre com `groups=g_tr`. A função `risco_cv` devolve a média e o erro-padrão das cinco estimativas.

In [ ]:
dobras = skm.GroupKFold(n_splits=5, shuffle=True, random_state=0)

def risco_cv(modelo, X_=None, cv=dobras):
    X_ = X_tr if X_ is None else X_
    s = -skm.cross_val_score(modelo, X_, y_tr, cv=cv, groups=g_tr,
                             scoring="neg_mean_squared_error")
    return s.mean(), s.std(ddof=1) / np.sqrt(len(s))

tabela_reg = {}

def registra(nome, modelo):
    m, ep = risco_cv(modelo)
    tabela_reg[nome] = (m, ep, modelo)
    print(f"{nome:40s} {m:8.2f} ± {ep:5.2f}")

## R3. Modelos lineares e regularização

**(a)** A referência é o preditor constante; depois, o MQO nas oito covariáveis originais.

In [ ]:
registra("constante (média do treino)", DummyRegressor())
registra("MQO, covariáveis originais", skl.LinearRegression())

mqo = skl.LinearRegression().fit(X_tr, y_tr)
print(pd.Series(mqo.coef_, index=covs).round(4))

Os sete componentes somam quase sempre a mesma massa por metro cúbico — é um volume de concreto —, então as colunas são quase linearmente dependentes com o intercepto. O número de condição mostra o tamanho do problema, e por que "aumentar a água mantendo o resto fixo" descreve uma mistura que quase não existe.

In [ ]:
soma = X_tr[componentes].sum(axis=1)
print(f"soma dos componentes: média {soma.mean():.0f}, desvio {soma.std():.1f} kg/m3")
A = np.column_stack([np.ones(len(X_tr)), X_tr.to_numpy()])
Z = np.column_stack([np.ones(len(X_tr)), StandardScaler().fit_transform(X_tr)])
print(f"cond(X'X), escala original: {np.linalg.cond(A.T @ A):.3g}")
print(f"cond(X'X), padronizado:     {np.linalg.cond(Z.T @ Z):.3g}")

**(b)** Duas covariáveis construídas: o logaritmo da idade, que lineariza o crescimento rápido seguido de platô, e a razão água/cimento (lei de Abrams). O modelo continua linear nos parâmetros.

In [ ]:
def construidas(Z):
    Z = Z.copy()
    Z["log_idade"] = np.log(Z["idade"])
    Z["agua_cimento"] = Z["agua"] / Z["cimento"]
    return Z.drop(columns=["idade"])

FE = FunctionTransformer(construidas)
registra("MQO, com log(idade) e água/cimento", make_pipeline(FE, skl.LinearRegression()))

**(c)** Expansão de grau 2 sobre as covariáveis construídas: $p$ passa de 9 para 54. O $\lambda$ é escolhido por validação cruzada em grupo, numa grade logarítmica, e a regra de um erro-padrão escolhe o maior $\lambda$ cujo risco fica a um erro-padrão do mínimo.

In [ ]:
def expande(estimador):
    return Pipeline([("fe", FE), ("poli", PolynomialFeatures(2, include_bias=False)),
                     ("sc", StandardScaler()), ("est", estimador)])

registra("MQO, grau 2", expande(skl.LinearRegression()))

grades = {"ridge": np.logspace(-3, 3, 31), "lasso": np.logspace(-3, 1, 31)}
buscas = {}
for nome, est in [("ridge", skl.Ridge()), ("lasso", skl.Lasso(max_iter=100_000))]:
    busca = skm.GridSearchCV(expande(est), {"est__alpha": grades[nome]}, cv=dobras,
                             scoring="neg_mean_squared_error")
    busca.fit(X_tr, y_tr, groups=g_tr)
    buscas[nome] = busca
    r = -busca.cv_results_["mean_test_score"]
    ep = busca.cv_results_["std_test_score"] / np.sqrt(5)
    i = int(np.argmin(r))
    ok = np.where(r <= r[i] + ep[i])[0]
    a_1ep = grades[nome][ok.max()]
    print(f"{nome}: alpha mínimo {grades[nome][i]:.4g} (risco {r[i]:.2f}); alpha 1-EP {a_1ep:.4g}")
    registra(f"{nome}, grau 2, alpha mínimo", expande(est.__class__(**{**est.get_params(), "alpha": grades[nome][i]})))
    registra(f"{nome}, grau 2, alpha 1-EP", expande(est.__class__(**{**est.get_params(), "alpha": a_1ep})))

A curva de validação cruzada dos dois métodos, com barras de um erro-padrão.

In [ ]:
fig, ax = subplots(1, 2, figsize=(12, 4))
for a_, nome in zip(ax, ["ridge", "lasso"]):
    b = buscas[nome]
    r = -b.cv_results_["mean_test_score"]; ep = b.cv_results_["std_test_score"] / np.sqrt(5)
    a_.errorbar(grades[nome], r, yerr=ep, fmt="o-", ms=3, capsize=2)
    a_.set_xscale("log"); a_.set_xlabel("alpha"); a_.set_ylabel("EQM de validação cruzada")
    a_.set_title(nome)
fig.tight_layout()

O caminho do Lasso e quantos coeficientes sobrevivem em cada escolha de $\lambda$.

In [ ]:
prep = Pipeline([("fe", FE), ("poli", PolynomialFeatures(2, include_bias=False)), ("sc", StandardScaler())])
Z_tr = prep.fit_transform(X_tr)
alfas, coefs, _ = skl.Lasso.path(Z_tr, y_tr - y_tr.mean(), alphas=np.logspace(-3, 1, 60), max_iter=100_000)
fig, ax = subplots(figsize=(8, 4))
ax.plot(alfas, coefs.T, lw=0.8)
ax.set_xscale("log"); ax.set_xlabel("alpha"); ax.set_ylabel("coeficiente (covariáveis padronizadas)")
for rotulo in ["lasso, grau 2, alpha mínimo", "lasso, grau 2, alpha 1-EP"]:
    m = tabela_reg[rotulo][2].fit(X_tr, y_tr)
    c = m.named_steps["est"].coef_
    ax.axvline(m.named_steps["est"].alpha, ls="--", c="k", lw=0.8)
    print(f"{rotulo}: {(c != 0).sum()} de {len(c)} coeficientes não nulos")

## R4. Validação cruzada à mão, LOOCV e bootstrap

**(a)** A validação cruzada em grupo, dobra a dobra, para o MQO com as covariáveis construídas. As dobras do `GroupKFold` não têm o mesmo tamanho, então a média simples dos cinco EQM (a do `cross_val_score` e do [ISLP]) e a média ponderada pelo tamanho (a das notas) diferem — pouco.

In [ ]:
F_tr = construidas(X_tr)
A = np.column_stack([np.ones(len(F_tr)), F_tr.to_numpy()])
eqm, tam = [], []
for a, b in dobras.split(A, y_tr, g_tr):
    beta = np.linalg.lstsq(A[a], y_tr[a], rcond=None)[0]
    eqm.append(np.mean((y_tr[b] - A[b] @ beta) ** 2)); tam.append(len(b))
eqm, tam = np.array(eqm), np.array(tam)
print("tamanhos:", tam)
print(f"média simples  {eqm.mean():.4f}")
print(f"média ponderada {(eqm * tam).sum() / tam.sum():.4f}")
print(f"cross_val_score {risco_cv(make_pipeline(FE, skl.LinearRegression()))[0]:.4f}")

**(b)** O atalho do LOOCV pela diagonal da matriz chapéu, contra o LOOCV feito com $n$ ajustes.

In [ ]:
beta = np.linalg.lstsq(A, y_tr, rcond=None)[0]
h = np.einsum("ij,ji->i", A, np.linalg.solve(A.T @ A, A.T))
residuo = y_tr - A @ beta
loo_atalho = np.mean((residuo / (1 - h)) ** 2)
loo_explicito = np.mean([
    (y_tr[i] - A[i] @ np.linalg.lstsq(np.delete(A, i, 0), np.delete(y_tr, i), rcond=None)[0]) ** 2
    for i in range(len(y_tr))])
print(f"LOOCV pelo atalho   {loo_atalho:.6f}")
print(f"LOOCV explícito     {loo_explicito:.6f}")
print(f"CV em grupo (5)     {risco_cv(make_pipeline(FE, skl.LinearRegression()))[0]:.4f}")
print(f"KFold por linha (5) {risco_cv(make_pipeline(FE, skl.LinearRegression()), cv=skm.KFold(5, shuffle=True, random_state=0))[0]:.4f}")

**(c)** Erro-padrão dos coeficientes do mesmo MQO: a fórmula clássica $\hat\sigma^2(\mathbb{X}^\top\mathbb{X})^{-1}$, o bootstrap que reamostra linhas e o que reamostra traços inteiros.

In [ ]:
s2 = residuo @ residuo / (len(y_tr) - A.shape[1])
ep_classico = np.sqrt(np.diag(s2 * np.linalg.inv(A.T @ A)))
rng = np.random.default_rng(2026)
B = 2000
por_traco = {t: np.where(g_tr == t)[0] for t in np.unique(g_tr)}
tracos = np.array(list(por_traco))
b_linha, b_traco = [], []
for _ in range(B):
    i = rng.integers(0, len(y_tr), len(y_tr))
    b_linha.append(np.linalg.lstsq(A[i], y_tr[i], rcond=None)[0])
    i = np.concatenate([por_traco[t] for t in rng.choice(tracos, len(tracos))])
    b_traco.append(np.linalg.lstsq(A[i], y_tr[i], rcond=None)[0])
eps = pd.DataFrame({"coeficiente": beta, "EP clássico": ep_classico,
                    "EP boot. linhas": np.std(b_linha, axis=0, ddof=1),
                    "EP boot. traços": np.std(b_traco, axis=0, ddof=1)},
                   index=["intercepto"] + F_tr.columns.tolist())
eps.round(4)

## R5. Métodos não paramétricos

**(a)** KNN padronizado, com $k$ de 1 a 30, e o mesmo KNN sem padronizar.

In [ ]:
ks = list(range(1, 31))
curvas = {}
for rotulo, modelo in [("padronizado", Pipeline([("sc", StandardScaler()), ("knn", KNeighborsRegressor())])),
                       ("padronizado + construídas", Pipeline([("fe", FE), ("sc", StandardScaler()), ("knn", KNeighborsRegressor())]))]:
    busca = skm.GridSearchCV(modelo, {"knn__n_neighbors": ks}, cv=dobras,
                             scoring="neg_mean_squared_error").fit(X_tr, y_tr, groups=g_tr)
    curvas[rotulo] = -busca.cv_results_["mean_test_score"]
    k = busca.best_params_["knn__n_neighbors"]
    registra(f"KNN {rotulo}, k={k}", modelo.set_params(knn__n_neighbors=k))
registra("KNN sem padronizar, k=5", KNeighborsRegressor(5))
fig, ax = subplots(figsize=(7, 4))
for rotulo, c in curvas.items():
    ax.plot(ks, c, "o-", ms=3, label=rotulo)
ax.set_xlabel("k"); ax.set_ylabel("EQM de validação cruzada"); ax.legend()

**(b)** *Splines* aditivos: cada covariável ganha uma base de *splines* cúbicos, e uma Ridge ajusta os coeficientes. Número de nós e $\lambda$ por validação cruzada, com os nós em duas posições: igualmente espaçados no intervalo de cada covariável (`"uniform"`, o padrão) ou nos quantis (`"quantile"`).

In [ ]:
modelo = Pipeline([("spl", SplineTransformer(degree=3)), ("sc", StandardScaler()), ("ridge", skl.Ridge())])
nos = [3, 4, 5, 6, 8, 10, 12, 15]
for posicao in ["uniform", "quantile"]:
    busca = skm.GridSearchCV(modelo, {"spl__knots": [posicao], "spl__n_knots": nos, "ridge__alpha": np.logspace(-3, 3, 13)},
                             cv=dobras, scoring="neg_mean_squared_error").fit(X_tr, y_tr, groups=g_tr)
    r = pd.DataFrame(busca.cv_results_).groupby("param_spl__n_knots")["mean_test_score"].max()
    print(posicao, "melhor EQM por número de nós:", (-r).round(1).to_dict())
    print("   escolha:", busca.best_params_)
registra("splines aditivos (nós nos quantis) + Ridge", modelo.set_params(**busca.best_params_))

## R6. Árvores e florestas

**(a)** A árvore cheia zera o erro de treino (a menos das réplicas, que ela não consegue separar) e vai mal na validação. A poda escolhe $\alpha$ numa grade logarítmica.

In [ ]:
cheia = DecisionTreeRegressor(random_state=0).fit(X_tr, y_tr)
print("árvore cheia: folhas", cheia.get_n_leaves(), " EQM de treino", round(mean_squared_error(y_tr, cheia.predict(X_tr)), 3))
registra("árvore cheia", DecisionTreeRegressor(random_state=0))
alfas_poda = np.r_[0, np.logspace(-3, 1.5, 28)]
busca = skm.GridSearchCV(DecisionTreeRegressor(random_state=0), {"ccp_alpha": alfas_poda}, cv=dobras,
                         scoring="neg_mean_squared_error").fit(X_tr, y_tr, groups=g_tr)
a_poda = busca.best_params_["ccp_alpha"]
registra(f"árvore podada, alpha={a_poda:.3g}", DecisionTreeRegressor(random_state=0, ccp_alpha=a_poda))
print("folhas da podada:", DecisionTreeRegressor(random_state=0, ccp_alpha=a_poda).fit(X_tr, y_tr).get_n_leaves())

fig, ax = subplots(figsize=(16, 6))
plot_tree(DecisionTreeRegressor(max_depth=3, random_state=0).fit(X_tr, y_tr), feature_names=covs,
          filled=True, impurity=False, precision=1, ax=ax);

**(b)** *Bagging* e florestas, com $m$ de 1 a 8 ($m=8=p$ é o *bagging*, e é o padrão do `RandomForestRegressor`).

In [ ]:
riscos_m = {}
for m in range(1, 9):
    riscos_m[m] = risco_cv(RandomForestRegressor(300, max_features=m, random_state=0, n_jobs=-1))
    print(f"m = {m}: {riscos_m[m][0]:6.2f} ± {riscos_m[m][1]:.2f}")
m_best = min(riscos_m, key=lambda k: riscos_m[k][0])
registra("bagging (m = p = 8)", RandomForestRegressor(300, max_features=None, random_state=0, n_jobs=-1))
registra(f"floresta, m={m_best}", RandomForestRegressor(300, max_features=m_best, random_state=0, n_jobs=-1))

**(c)** O risco contra o número de árvores $B$.

In [ ]:
Bs = [1, 2, 5, 10, 25, 50, 100, 200, 400]
rB = [risco_cv(RandomForestRegressor(b, max_features=m_best, random_state=0, n_jobs=-1))[0] for b in Bs]
fig, ax = subplots(figsize=(7, 4))
ax.plot(Bs, rB, "o-"); ax.set_xscale("log"); ax.set_xlabel("B (número de árvores)"); ax.set_ylabel("EQM de validação cruzada")
print(dict(zip(Bs, np.round(rB, 2))))

## R7. Escala e vazamento

**(a)** A idade em horas em vez de dias: quem muda?

In [ ]:
X_horas = X_tr.copy(); X_horas["idade"] = X_horas["idade"] * 24
for nome, modelo in [("MQO", skl.LinearRegression()),
                     ("Lasso alpha=1, padronizado", make_pipeline(StandardScaler(), skl.Lasso(alpha=1.0))),
                     ("KNN k=5, sem padronizar", KNeighborsRegressor(5)),
                     ("KNN k=5, padronizado", make_pipeline(StandardScaler(), KNeighborsRegressor(5))),
                     ("árvore podada", DecisionTreeRegressor(random_state=0, ccp_alpha=a_poda)),
                     ("floresta", RandomForestRegressor(300, max_features=m_best, random_state=0, n_jobs=-1))]:
    print(f"{nome:28s} dias {risco_cv(modelo)[0]:8.3f}   horas {risco_cv(modelo, X_horas)[0]:8.3f}")

**(b)** O vazamento por unidade: `KFold` por linha contra `GroupKFold` por traço, no mesmo treino.

In [ ]:
kf = skm.KFold(5, shuffle=True, random_state=0)
for nome in ["MQO, com log(idade) e água/cimento", "lasso, grau 2, alpha mínimo",
             "splines aditivos (nós nos quantis) + Ridge", f"floresta, m={m_best}",
             [k for k in tabela_reg if k.startswith("KNN padronizado + ")][0]]:
    modelo = tabela_reg[nome][2]
    print(f"{nome[:40]:40s} KFold {risco_cv(modelo, cv=kf)[0]:7.2f}   GroupKFold {risco_cv(modelo)[0]:7.2f}")
for k in [1, 5]:
    for rotulo, modelo in [("sem padronizar", KNeighborsRegressor(k)), ("padronizado", make_pipeline(StandardScaler(), KNeighborsRegressor(k)))]:
        print(f"KNN k={k} {rotulo:28s} KFold {risco_cv(modelo, cv=kf)[0]:7.2f}   GroupKFold {risco_cv(modelo)[0]:7.2f}")

## R8. Escolha final e teste

A tabela da validação cruzada, ordenada pelo risco.

In [ ]:
tab = pd.DataFrame({k: v[:2] for k, v in tabela_reg.items()}, index=["EQM (CV)", "EP"]).T.sort_values("EQM (CV)")
tab.round(2)

Os *splines* aditivos com nós nos quantis e a floresta empatam dentro de um erro-padrão. Fica o de menor risco estimado, os *splines* — que têm, de quebra, a vantagem de ser aditivos: o efeito de cada covariável é uma curva que se pode desenhar. O modelo é reajustado em todo o treino e medido **uma vez** no teste, com o intervalo da Aula 01 e o $R^2$.

In [ ]:
escolhido = tab.index[0]
print("escolhido:", escolhido)
final = tabela_reg[escolhido][2].fit(X_tr, y_tr)
w = (y_te - final.predict(X_te)) ** 2
print(f"EQM de teste {w.mean():.2f}  IC95% ({w.mean() - 2 * w.std(ddof=1) / np.sqrt(len(w)):.2f}; {w.mean() + 2 * w.std(ddof=1) / np.sqrt(len(w)):.2f})")
print(f"R2 de teste {1 - w.mean() / np.var(y_te):.3f}   variância de y no treino {np.var(y_tr):.1f}, no teste {np.var(y_te):.1f}")

Só depois de escolhido o modelo, os outros finalistas no mesmo teste — para leitura, não para trocar a escolha.

In [ ]:
for nome in ["constante (média do treino)", "MQO, com log(idade) e água/cimento", "lasso, grau 2, alpha 1-EP",
             "splines aditivos (nós nos quantis) + Ridge", f"árvore podada, alpha={a_poda:.3g}",
             [k for k in tabela_reg if k.startswith("KNN padronizado + ")][0], f"floresta, m={m_best}"]:
    w = (y_te - tabela_reg[nome][2].fit(X_tr, y_tr).predict(X_te)) ** 2
    print(f"{nome[:45]:45s} {w.mean():7.2f} ± {2 * w.std(ddof=1) / np.sqrt(len(w)):5.2f}")

Os *splines*, escolhidos pela validação, ficaram atrás da floresta no teste. O diagnóstico dos maiores erros deles: as observações em que uma covariável cai fora do intervalo visto no treino. Um traço do teste tem menos água e mais agregado miúdo do que qualquer traço do treino, e a base cúbica extrapola; a floresta, que é constante fora dos dados, não. A escolha não muda por causa disso — o teste mede, não escolhe —, mas o relatório tem de contar.

In [ ]:
fora = ((X_te < X_tr.min()) | (X_te > X_tr.max())).any(axis=1).to_numpy()
spl = tabela_reg["splines aditivos (nós nos quantis) + Ridge"][2].fit(X_tr, y_tr)
flo = tabela_reg[f"floresta, m={m_best}"][2].fit(X_tr, y_tr)
diag = X_te.assign(y=y_te, erro2_splines=(y_te - spl.predict(X_te)) ** 2,
                   erro2_floresta=(y_te - flo.predict(X_te)) ** 2, fora_do_treino=fora)
print("observações de teste fora do intervalo do treino:", fora.sum())
for nome_ in ["erro2_splines", "erro2_floresta"]:
    print(f"{nome_}: com todas {diag[nome_].mean():.2f}, sem as de fora {diag.loc[~fora, nome_].mean():.2f}")
diag.sort_values("erro2_splines", ascending=False).head(6).round(1)

# Parte II — Classificação: inadimplência no cartão de crédito

## C1. Conhecendo os dados

In [ ]:
_nome = "credito.csv"

# procura em dois lugares, sem baixar nada da internet: a pasta deste
# notebook primeiro ou então ../recursos/dados/
_lugares = [_nome, os.path.join("..", "recursos", "dados", _nome)]
_caminho = next((c for c in _lugares if os.path.exists(c)), None)

if _caminho is None:
    raise FileNotFoundError(
        f"Não encontrei {_nome}. Procurei em: {', '.join(_lugares)}. "
        "Coloque o arquivo na mesma pasta deste notebook.")

credito = pd.read_csv(_caminho)
print(credito.shape, " prevalência:", round(credito["default"].mean(), 4))
for v in ["SEX", "EDUCATION", "MARRIAGE", "PAY_0"]:
    print(v, credito[v].value_counts().sort_index().to_dict())

**(b)** O `ID` é identificador e sai. As categorias não documentadas de `EDUCATION` (0, 5 e 6) e de `MARRIAGE` (0) vão para "outros" (4 e 3, respectivamente) — uma escolha entre várias razoáveis; o que não é razoável é tratá-las como números.

In [ ]:
credito["EDUCATION"] = credito["EDUCATION"].replace({0: 4, 5: 4, 6: 4})
credito["MARRIAGE"] = credito["MARRIAGE"].replace({0: 3})
Xc = credito.drop(columns=["ID", "default"])
yc = credito["default"].to_numpy()
categoricas = ["SEX", "EDUCATION", "MARRIAGE"]
atrasos = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
print(Xc[[f"BILL_AMT{i}" for i in range(1, 7)]].corr().round(2))
print("inadimplência por decil do ID:", credito.groupby(pd.qcut(credito["ID"], 10), observed=True)["default"].mean().round(3).tolist())
print("linhas idênticas a outra, fora o ID:", credito.drop(columns="ID").duplicated().sum())

## C2. Protocolo e custos

**(a)** Teste estratificado, com a semente do enunciado.

In [ ]:
Xc_tr, Xc_te, yc_tr, yc_te = skm.train_test_split(Xc, yc, test_size=0.25, stratify=yc, random_state=2026)
dobras_c = skm.StratifiedKFold(5, shuffle=True, random_state=0)
print(len(yc_tr), len(yc_te), round(yc_tr.mean(), 4), round(yc_te.mean(), 4))
print("acurácia de 'ninguém fica inadimplente' no treino:", round(1 - yc_tr.mean(), 4))

**(b)** Custos: um falso negativo custa $c_{FN}=5$, um falso positivo $c_{FP}=1$. O corte ótimo para probabilidades verdadeiras é $1/6$, e o custo médio das duas regras triviais sai direto da prevalência.

In [ ]:
C_FN, C_FP = 5, 1
corte = C_FP / (C_FP + C_FN)

def custo(y, p, k=corte):
    return np.mean(C_FN * ((p < k) & (y == 1)) + C_FP * ((p >= k) & (y == 0)))

print("corte ótimo:", round(corte, 4))
print("custo de não contatar ninguém:", round(custo(yc_tr, np.zeros(len(yc_tr))), 4))
print("custo de contatar todos:     ", round(custo(yc_tr, np.ones(len(yc_tr))), 4))

## C3. Logística e modelos generativos

Dois pré-processamentos: as categóricas em *dummies* e o resto padronizado; e o mesmo com os `PAY_*` também em *dummies*.

In [ ]:
def prep(cats):
    nums = [c for c in Xc.columns if c not in cats]
    return ColumnTransformer([("num", StandardScaler(), nums),
                              ("cat", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), cats)])

oof = {}          # probabilidades fora da dobra, no treino

def fora_da_dobra(nome, modelo):
    t = time.time()
    p = skm.cross_val_predict(modelo, Xc_tr, yc_tr, cv=dobras_c, method="predict_proba")[:, 1]
    oof[nome] = (p, modelo)
    print(f"{nome:34s} AUC {roc_auc_score(yc_tr, p):.4f}  AP {average_precision_score(yc_tr, p):.4f}  "
          f"Brier {brier_score_loss(yc_tr, p):.4f}  custo@1/6 {custo(yc_tr, p):.4f}  [{time.time() - t:.0f}s]")

for rotulo, cats in [("PAY numérico", categoricas), ("PAY em dummies", categoricas + atrasos)]:
    busca = skm.GridSearchCV(Pipeline([("prep", prep(cats)), ("log", skl.LogisticRegression(max_iter=5000))]),
                             {"log__C": np.logspace(-4, 2, 13)}, cv=dobras_c, scoring="neg_log_loss").fit(Xc_tr, yc_tr)
    print(rotulo, busca.best_params_)
    fora_da_dobra(f"logística, {rotulo}", busca.best_estimator_)

**(a)** Razões de chances da logística com `PAY_*` numérico: como as covariáveis numéricas estão padronizadas, $e^{\beta_j}$ é o fator na chance por **um desvio-padrão** a mais.

In [ ]:
log_num = oof["logística, PAY numérico"][1].fit(Xc_tr, yc_tr)
rc = pd.Series(np.exp(log_num.named_steps["log"].coef_[0]), index=log_num.named_steps["prep"].get_feature_names_out())
print(rc.loc[["num__PAY_0", "num__LIMIT_BAL", "num__BILL_AMT1", "num__BILL_AMT2", "cat__SEX_2"]].round(3))
print("desvio-padrão de PAY_0 e LIMIT_BAL:", round(Xc_tr["PAY_0"].std(), 3), round(Xc_tr["LIMIT_BAL"].std()))

**(c)** Bayes ingênuo, LDA e QDA. O QDA sem regularização recusa o ajuste; a mensagem do `scikit-learn` diz por quê.

In [ ]:
fora_da_dobra("Bayes ingênuo gaussiano", make_pipeline(prep(categoricas), GaussianNB()))
fora_da_dobra("LDA", make_pipeline(prep(categoricas), LinearDiscriminantAnalysis()))
try:
    make_pipeline(prep(categoricas), QuadraticDiscriminantAnalysis()).fit(Xc_tr, yc_tr)
except Exception as e:
    print(type(e).__name__, ":", e)
for rp in [0.01, 0.1, 0.3, 0.5, 0.8]:
    p = skm.cross_val_predict(make_pipeline(prep(categoricas), QuadraticDiscriminantAnalysis(reg_param=rp)),
                              Xc_tr, yc_tr, cv=dobras_c, method="predict_proba")[:, 1]
    print(f"QDA reg_param={rp}: AUC {roc_auc_score(yc_tr, p):.4f}  log-loss {log_loss(yc_tr, p):.4f}")
fora_da_dobra("QDA, reg_param=0,5", make_pipeline(prep(categoricas), QuadraticDiscriminantAnalysis(reg_param=0.5)))
p_ = prep(categoricas).fit_transform(Xc_tr).shape[1]
print("p depois das dummies:", p_, "  parâmetros de covariância: LDA", p_ * (p_ + 1) // 2, " QDA", 2 * p_ * (p_ + 1) // 2)

## C4. Métricas, custos e calibração

**(a)** Matriz de confusão da logística (com `PAY_*` em *dummies*) no corte 0,5, fora da dobra.

In [ ]:
p_log = oof["logística, PAY em dummies"][0]
vn, fp, fn, vp = confusion_matrix(yc_tr, p_log >= 0.5).ravel()
print(f"VN {vn}  FP {fp}  FN {fn}  VP {vp}")
S, E, P, VPN = vp / (vp + fn), vn / (vn + fp), vp / (vp + fp), vn / (vn + fn)
print(f"acurácia {(vp + vn) / len(yc_tr):.4f}  sensibilidade {S:.4f}  especificidade {E:.4f}  "
      f"precisão {P:.4f}  VPN {VPN:.4f}  F1 {2 * P * S / (P + S):.4f}")

**(b)** e **(c)** ROC, precisão--*recall* e calibração, para a logística, o Bayes ingênuo e o Bayes ingênuo calibrado (isotônica, com validação cruzada interna).

In [ ]:
fora_da_dobra("Bayes ingênuo calibrado", CalibratedClassifierCV(make_pipeline(prep(categoricas), GaussianNB()), method="isotonic", cv=5))
p_nb = oof["Bayes ingênuo gaussiano"][0]
print("fração de probabilidades do NB acima de 0,99:", round((p_nb > 0.99).mean(), 4), " abaixo de 0,01:", round((p_nb < 0.01).mean(), 4))
fig, ax = subplots(1, 3, figsize=(16, 4.5))
for nome in ["logística, PAY em dummies", "Bayes ingênuo gaussiano", "Bayes ingênuo calibrado", "LDA"]:
    p = oof[nome][0]
    fpr, tpr, _ = roc_curve(yc_tr, p); ax[0].plot(fpr, tpr, label=nome)
    pr, rc_, _ = precision_recall_curve(yc_tr, p); ax[1].plot(rc_, pr, label=nome)
    fo, mp = calibration_curve(yc_tr, p, n_bins=10, strategy="quantile"); ax[2].plot(mp, fo, "o-", ms=3, label=nome)
ax[0].plot([0, 1], [0, 1], "k:"); ax[0].set_xlabel("1 - especificidade"); ax[0].set_ylabel("sensibilidade")
ax[1].axhline(yc_tr.mean(), c="k", ls=":"); ax[1].set_xlabel("recall"); ax[1].set_ylabel("precisão")
ax[2].plot([0, 1], [0, 1], "k:"); ax[2].set_xlabel("probabilidade predita"); ax[2].set_ylabel("frequência observada")
ax[0].legend(fontsize=8); fig.tight_layout()

**(d)** O custo no corte teórico $1/6$ contra o melhor corte encontrado nas próprias probabilidades fora da dobra, para cada modelo. E a alternativa de reponderar as classes.

In [ ]:
fora_da_dobra("logística, class_weight='balanced'",
              Pipeline([("prep", prep(categoricas + atrasos)),
                        ("log", skl.LogisticRegression(max_iter=5000, C=oof["logística, PAY em dummies"][1].named_steps["log"].C,
                                                       class_weight="balanced"))]))
cortes = np.linspace(0.02, 0.95, 94)
for nome, (p, _) in oof.items():
    cs = [custo(yc_tr, p, k) for k in cortes]
    print(f"{nome:36s} custo@1/6 {custo(yc_tr, p):.4f}   melhor corte {cortes[int(np.argmin(cs))]:.2f} ({min(cs):.4f})   custo@0,5 {custo(yc_tr, p, 0.5):.4f}")

Com `class_weight="balanced"`, cada classe pesa o inverso da sua frequência, e a chance estimada fica multiplicada, aproximadamente, por $n_0/n_1$. O corte $1/6$ na probabilidade verdadeira (chance $1/5$) corresponde, no modelo reponderado, à chance $\frac15\cdot\frac{n_0}{n_1}$ — e o corte equivalente sai da conta, sem olhar os dados de novo.

In [ ]:
n1, n0 = yc_tr.sum(), len(yc_tr) - yc_tr.sum()
chance = (corte / (1 - corte)) * n0 / n1
print(f"n0/n1 = {n0 / n1:.3f};  corte equivalente = {chance / (1 + chance):.3f}")
p_bal = oof["logística, class_weight='balanced'"][0]
print(f"custo da reponderada nesse corte: {custo(yc_tr, p_bal, chance / (1 + chance)):.4f}")

## C5. SVM, numa subamostra

Subamostra estratificada de 5000 observações do treino, com a semente do enunciado.

In [ ]:
i_sub = skm.train_test_split(np.arange(len(yc_tr)), train_size=5000, stratify=yc_tr, random_state=2026)[0]
Xs, ys = Xc_tr.iloc[i_sub], yc_tr[i_sub]
t = time.time()
busca_svm = skm.GridSearchCV(make_pipeline(prep(categoricas + atrasos), SVC(kernel="rbf")),
                             {"svc__C": [0.1, 1, 10, 100], "svc__gamma": [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]},
                             cv=dobras_c, scoring="roc_auc", n_jobs=-1).fit(Xs, ys)
print("RBF:", busca_svm.best_params_, round(busca_svm.best_score_, 4), f"[{time.time() - t:.0f}s]")
print(pd.DataFrame(busca_svm.cv_results_).pivot(index="param_svc__C", columns="param_svc__gamma", values="mean_test_score").round(4))
busca_lin = skm.GridSearchCV(make_pipeline(prep(categoricas + atrasos), SVC(kernel="linear")), {"svc__C": [0.01, 0.1, 1]},
                             cv=dobras_c, scoring="roc_auc", n_jobs=-1).fit(Xs, ys)
print("linear:", busca_lin.best_params_, round(busca_lin.best_score_, 4))
log_sub = -1
log_sub = skm.cross_val_score(Pipeline([("prep", prep(categoricas + atrasos)), ("log", skl.LogisticRegression(max_iter=5000, C=0.1))]),
                              Xs, ys, cv=dobras_c, scoring="roc_auc").mean()
print("logística na mesma subamostra:", round(log_sub, 4))
for C in [0.1, 1, 10, 100]:
    s = make_pipeline(prep(categoricas + atrasos), SVC(kernel="rbf", C=C, gamma=busca_svm.best_params_["svc__gamma"])).fit(Xs, ys)
    print(f"C={C}: vetores de suporte {s.named_steps['svc'].n_support_.sum()} de {len(ys)}")

## C6. KNN, árvores e florestas

In [ ]:
busca = skm.GridSearchCV(make_pipeline(prep(categoricas + atrasos), KNeighborsClassifier()),
                         {"kneighborsclassifier__n_neighbors": [1, 5, 15, 31, 51, 101, 151, 201, 301, 401]},
                         cv=dobras_c, scoring="roc_auc").fit(Xc_tr, yc_tr)
print(dict(zip(busca.cv_results_["param_kneighborsclassifier__n_neighbors"], busca.cv_results_["mean_test_score"].round(4))))
k_knn = busca.best_params_["kneighborsclassifier__n_neighbors"]
fora_da_dobra(f"KNN, k={k_knn}", make_pipeline(prep(categoricas + atrasos), KNeighborsClassifier(k_knn)))

**(b)** Gini contra entropia nos três primeiros níveis, e a poda por validação cruzada.

In [ ]:
for crit in ["gini", "entropy"]:
    a = DecisionTreeClassifier(max_depth=3, criterion=crit, random_state=0).fit(Xc_tr, yc_tr)
    print(crit, [(Xc.columns[f], round(s, 1)) for f, s in zip(a.tree_.feature, a.tree_.threshold) if f >= 0])
busca = skm.GridSearchCV(DecisionTreeClassifier(random_state=0), {"ccp_alpha": np.r_[0, np.logspace(-5, -2, 16)]},
                         cv=dobras_c, scoring="neg_log_loss").fit(Xc_tr, yc_tr)
a_c = busca.best_params_["ccp_alpha"]
arv_c = DecisionTreeClassifier(random_state=0, ccp_alpha=a_c).fit(Xc_tr, yc_tr)
print("alpha", round(a_c, 6), " folhas", arv_c.get_n_leaves())
fora_da_dobra("árvore cheia", DecisionTreeClassifier(random_state=0))
fora_da_dobra("árvore podada", DecisionTreeClassifier(random_state=0, ccp_alpha=a_c))
fig, ax = subplots(figsize=(16, 6))
plot_tree(DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xc_tr, yc_tr), feature_names=list(Xc.columns),
          class_names=["adimplente", "inadimplente"], filled=True, precision=2, ax=ax);

**(c)** Florestas com $m=\sqrt p$ (o padrão), $m\approx p/3$ e $m=p$ (*bagging*), folhas com no mínimo 5 observações; e o AdaBoost, que o enunciado deixa opcional.

In [ ]:
for rotulo, mf in [("sqrt(p)", "sqrt"), ("p/3", 0.33), ("p (bagging)", None)]:
    fora_da_dobra(f"floresta, m={rotulo}", RandomForestClassifier(200, max_features=mf, min_samples_leaf=5, n_jobs=-1, random_state=0))
fora_da_dobra("floresta, m=sqrt(p), folhas cheias", RandomForestClassifier(200, n_jobs=-1, random_state=0))
fora_da_dobra("AdaBoost (200 tocos)", AdaBoostClassifier(n_estimators=200, random_state=0))
p_ada = oof["AdaBoost (200 tocos)"][0]
print("probabilidades do AdaBoost: mínimo", p_ada.min().round(3), " máximo", p_ada.max().round(3))

## C7. Escolha final e teste

A tabela, fora da dobra no treino.

In [ ]:
linhas = {}
for nome, (p, _) in oof.items():
    cs = [custo(yc_tr, p, k) for k in cortes]
    linhas[nome] = [roc_auc_score(yc_tr, p), average_precision_score(yc_tr, p), brier_score_loss(yc_tr, p),
                    custo(yc_tr, p), cortes[int(np.argmin(cs))], min(cs)]
tab_c = pd.DataFrame(linhas, index=["AUC", "AP", "Brier", "custo@1/6", "melhor corte", "custo mínimo"]).T
tab_c.sort_values("custo@1/6").round(4)

O critério é o custo no corte $1/6$. O modelo de menor custo é reajustado no treino inteiro e medido uma vez no teste, no corte $1/6$ (fixado antes de olhar o teste) — o que só se justifica porque o melhor corte dele, fora da dobra, está perto de $1/6$: as probabilidades dele são razoavelmente calibradas.

In [ ]:
escolhido_c = tab_c["custo@1/6"].idxmin()
print("escolhido:", escolhido_c)
modelo_final = oof[escolhido_c][1].fit(Xc_tr, yc_tr)
q = modelo_final.predict_proba(Xc_te)[:, 1]
vn, fp, fn, vp = confusion_matrix(yc_te, q >= corte).ravel()
print(f"teste: VN {vn} FP {fp} FN {fn} VP {vp}")
print(f"custo médio {custo(yc_te, q):.4f}  (contatar todos: {custo(yc_te, np.ones(len(yc_te))):.4f})")
print(f"AUC {roc_auc_score(yc_te, q):.4f}  Brier {brier_score_loss(yc_te, q):.4f}  sensibilidade {vp / (vp + fn):.4f}  contatados {(q >= corte).mean():.4f}")

O *bootstrap* pareado no teste: a mesma reamostra dos clientes para os dois modelos, e a diferença de AUC e de custo entre o modelo escolhido e a logística com `PAY_*` em *dummies*. Pareado porque os dois são avaliados nos mesmos clientes, e os erros deles andam juntos.

In [ ]:
q_log = oof["logística, PAY em dummies"][1].fit(Xc_tr, yc_tr).predict_proba(Xc_te)[:, 1]
rng = np.random.default_rng(2026)
dif_auc, dif_custo = [], []
for _ in range(1000):
    i = rng.integers(0, len(yc_te), len(yc_te))
    dif_auc.append(roc_auc_score(yc_te[i], q[i]) - roc_auc_score(yc_te[i], q_log[i]))
    dif_custo.append(custo(yc_te[i], q[i]) - custo(yc_te[i], q_log[i]))
print(f"AUC:   escolhido {roc_auc_score(yc_te, q):.4f}, logística {roc_auc_score(yc_te, q_log):.4f}; "
      f"diferença IC95% ({np.percentile(dif_auc, 2.5):.4f}; {np.percentile(dif_auc, 97.5):.4f})")
print(f"custo: escolhido {custo(yc_te, q):.4f}, logística {custo(yc_te, q_log):.4f}; "
      f"diferença IC95% ({np.percentile(dif_custo, 2.5):.4f}; {np.percentile(dif_custo, 97.5):.4f})")
print(f"tempo total: {time.time() - inicio:.0f} s")